In [1]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

/Users/shukanchavda/RAG_INT/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
folder = "HR_DOC"

docs = []

for file in os.listdir(folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        docs.extend(loader.load())

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

splits = text_splitter.split_documents(docs)

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

/var/folders/j0/2l85bcxs7zl4h52g8cy7yllm0000gn/T/ipykernel_6894/2394043513.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2694.54it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [6]:
vectorstore.similarity_search("What is the leave policy?")

[Document(id='8031a9af-19e9-4c6b-af00-9738dbb1c35d', metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2019-01-25T16:37:28+00:00', 'moddate': '2019-01-25T16:37:28+00:00', 'source': 'HR_DOC/USA_Employee_Handbook-Freely_Available.pdf', 'total_pages': 34, 'page': 5, 'page_label': '6'}, page_content='networks and external networks as potential resources for referred candidates. \n \nKeep in mind that rewards may be subject to taxation. Please contact HR or our referral \nprogram manager for more information. \nAttendance \nWe expect you to be present during your scheduled working hours. If you face an \nemergency that prevents you from coming to work one day, contact your manager as \nsoon as possible. We will excuse unreported absences in cases of [serious accidents, \nacute medical emergencies.] But, whenever possible, we should know when you won’t \nbe coming in. \nWorkplace policies \nThis section describes policies that apply to everyone at our company: empl

In [7]:
llm = Ollama(model="mistral")

/var/folders/j0/2l85bcxs7zl4h52g8cy7yllm0000gn/T/ipykernel_6894/86901005.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral")


In [8]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the context below.

Context:
{context}

Question:
{question}
""")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm

)

In [9]:
rag_chain.invoke("What is the leave policy?")

" The leave policy, as described in the provided context, includes the following:\n\n1. Unreported absences are excused in cases of serious accidents or acute medical emergencies.\n2. Employees can request an unpaid leave extension of up to two months. Contact HR as soon as possible to arrange this.\n3. The company offers paid maternity and paternity leave for up to three months. If local or national law stipulates longer leave, the company will follow the law.\n4. New parents (either through childbirth or adoption) can take up to three months of paid leave. They should give at least three months' notice before their leave begins.\n5. Pregnant women can take part of their leave before labor if they suffer complications during childbirth or have other issues, and can ask for an unpaid leave extension of up to two months. Contact HR as soon as possible to arrange this.\n\nAdditionally, the company offers remote working/flexible hours, onsite/external paid day care, and lactation rooms fo

In [12]:
rag_chain.invoke("How many paid leaves do i get?")

' Based on the provided context, an employee receives 20 days of Paid Time Off (PTO) per year.'

In [10]:
!which python

/Users/shukanchavda/RAG_INT/venv/bin/python
